# BioRAG-Hybrid: embedding generation on Kaggle GPU

This notebook generates frozen BioBERT and MedCPT embeddings. It does not train models.

## Kaggle setup
1. Create a Kaggle Dataset and upload `biorag_kaggle_bundle.zip` from this project.
2. Create a Kaggle Notebook, attach that dataset with **Add Input**, and select a GPU accelerator in Notebook Settings.
3. Turn **Internet** on in Notebook Settings so the pretrained model weights can be downloaded.
4. Import this notebook, then run its cells in order.

The attached input is read-only. The notebook extracts it under `/kaggle/working`, writes the generated caches there, and packages them into an output ZIP. After the run, save a notebook version and download `biorag_embedding_caches.zip` from its Output files. No Google Drive mount or API key is needed.

In [ ]:
import torch

gpu_count = torch.cuda.device_count()
if gpu_count < 2:
    raise RuntimeError(f'Expected two visible GPUs, found {gpu_count}. In Kaggle Notebook Settings select GPU T4 x2 (if available), then restart the session.')
print('Visible GPUs:', [torch.cuda.get_device_name(i) for i in range(gpu_count)])
print('PyTorch:', torch.__version__)

In [ ]:
from pathlib import Path
import shutil
import zipfile

INPUT_ROOT = Path('/kaggle/input')
BUNDLE_DIR = Path('/kaggle/working/biorag_project')
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
# Kaggle may expose an attached dataset as extracted files or as a ZIP.
matches = list(INPUT_ROOT.rglob('encode_embeddings.py'))
matches = [path for path in matches if path.parent.name == 'tools']
if matches:
    if len(matches) != 1:
        raise FileNotFoundError(f'Found multiple encoder scripts: {matches}')
    source_root = matches[0].parents[1]
    PROJECT_ROOT = BUNDLE_DIR / source_root.name
    shutil.copytree(source_root, PROJECT_ROOT, dirs_exist_ok=True)
else:
    bundles = list(INPUT_ROOT.rglob('*.zip'))
    if not bundles:
        visible = [str(path.relative_to(INPUT_ROOT)) for path in INPUT_ROOT.rglob('*') if path.is_file()]
        raise FileNotFoundError('No ZIP or tools/encode_embeddings.py found under /kaggle/input. Attached files: ' + repr(visible[:40]))
    preferred = [path for path in bundles if 'biorag' in path.name.lower()]
    bundle = preferred[0] if len(preferred) == 1 else (bundles[0] if len(bundles) == 1 else None)
    if bundle is None:
        raise FileNotFoundError(f'Found multiple ZIPs; keep the BioRAG bundle attached by itself: {bundles}')
    print('Extracting attached bundle:', bundle)
    with zipfile.ZipFile(bundle) as archive:
        archive.extractall(BUNDLE_DIR)
    matches = list(BUNDLE_DIR.rglob('tools/encode_embeddings.py'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Bundle must contain exactly one tools/encode_embeddings.py; found {matches}')
    PROJECT_ROOT = matches[0].parents[1]
DATA_DIR = PROJECT_ROOT / 'data/official'
print('Attached input root:', INPUT_ROOT)
print('Project files extracted to:', PROJECT_ROOT)
print('Question file exists:', (DATA_DIR / 'training11b.json').is_file())
print('PubMed corpus exists:', (DATA_DIR / 'pubmed_abstracts.json').is_file())

In [ ]:
# Keep Kaggle's preinstalled PyTorch/CUDA build. Install encoder/cache dependencies.
%pip -q install 'transformers>=4.40,<5' 'numpy>=1.24,<3'

In [ ]:
from pathlib import Path
import subprocess

OUTPUT_DIR = Path('/kaggle/working/biorag_embeddings')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 16  # DataParallel splits this across both GPUs (8 examples per T4). Lower to 8 if memory is tight.
CHECKPOINT_EVERY = 512  # Resume from the last flushed batch if a cell is interrupted.

def encode(model, kind, input_path, output_name, max_length):
    command = [
        'python', str(PROJECT_ROOT / 'tools/encode_embeddings.py'),
        '--kind', kind, '--model', model,
        '--input', str(input_path), '--output', str(OUTPUT_DIR / output_name),
        '--batch-size', str(BATCH_SIZE), '--max-length', str(max_length), '--device', 'cuda', '--multi-gpu',
        '--checkpoint-every', str(CHECKPOINT_EVERY),
    ]
    print('Running:', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

print('Output directory:', OUTPUT_DIR)

In [ ]:
from IPython.display import FileLink, display
import zipfile

def publish_checkpoint(names, archive_name):
    missing = [name for name in names if not (OUTPUT_DIR / name).is_file()]
    if missing:
        raise FileNotFoundError(f'Cannot package checkpoint; missing files: {missing}')
    archive_path = Path('/kaggle/working') / archive_name
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_STORED) as archive:
        for name in names:
            archive.write(OUTPUT_DIR / name, arcname=name)
    print('Checkpoint ready. Click the link to download a local copy:', archive_path)
    display(FileLink(str(archive_path)))

BIOBERT_FILES = ['biobert_documents.npz', 'biobert_documents.npz.json', 'biobert_queries.npz', 'biobert_queries.npz.json']
MEDCPT_FILES = ['medcpt_documents.npz', 'medcpt_documents.npz.json', 'medcpt_queries.npz', 'medcpt_queries.npz.json']

## Generate BioBERT caches

BioBERT uses masked mean pooling. Document text combines title and abstract. Defaults are 512 tokens per document and 64 per question. Batches are split across the two visible GPUs.

In [ ]:
encode('biobert', 'documents', DATA_DIR / 'pubmed_abstracts.json', 'biobert_documents.npz', 512)
encode('biobert', 'questions', DATA_DIR / 'training11b.json', 'biobert_queries.npz', 64)
publish_checkpoint(BIOBERT_FILES, 'biorag_biobert_checkpoint.zip')

## Generate MedCPT caches

MedCPT uses its separate pretrained query and article encoders, with `[CLS]` vectors. The article encoder gets title/abstract pairs.

In [ ]:
encode('medcpt-article', 'documents', DATA_DIR / 'pubmed_abstracts.json', 'medcpt_documents.npz', 512)
encode('medcpt-query', 'questions', DATA_DIR / 'training11b.json', 'medcpt_queries.npz', 64)
publish_checkpoint(MEDCPT_FILES, 'biorag_medcpt_checkpoint.zip')

## Package the output caches

The model-specific checkpoint ZIP links above let you save a local copy after each completed model. This final archive contains all four caches. Click its download link, and also save a notebook version to retain Kaggle's output.

In [ ]:
cache_names = [
    'biobert_documents.npz', 'biobert_documents.npz.json',
    'biobert_queries.npz', 'biobert_queries.npz.json',
    'medcpt_documents.npz', 'medcpt_documents.npz.json',
    'medcpt_queries.npz', 'medcpt_queries.npz.json',
]
missing = [name for name in cache_names if not (OUTPUT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing cache outputs: {missing}')
CACHE_ZIP = Path('/kaggle/working/biorag_embedding_caches.zip')
with zipfile.ZipFile(CACHE_ZIP, 'w', compression=zipfile.ZIP_STORED) as archive:
    for name in cache_names:
        archive.write(OUTPUT_DIR / name, arcname=name)
print('Created:', CACHE_ZIP)
print('Size (MB):', round(CACHE_ZIP.stat().st_size / (1024 * 1024), 1))
print('Click to download all caches:')
display(FileLink(str(CACHE_ZIP)))

## Finish locally

Download `biorag_embedding_caches.zip` from this Kaggle notebook's Output files and extract it into `data/official/embeddings/` in the project. Then run the CPU evaluation command in `data/official/embeddings/README.md`. The validation split selects fusion weights; the test split is evaluated afterward.